In [4]:
import pandas as pd
import glob
import os


In [1]:
import os
print(os.listdir("data"))
print(os.listdir("data/enrolment"))


['.ipynb_checkpoints', 'biometric', 'demographic', 'enrolment']
['api_data_aadhar_enrolment_0_500000.csv', 'api_data_aadhar_enrolment_1000000_1006029.csv', 'api_data_aadhar_enrolment_500000_1000000.csv']


In [2]:
def load_and_merge_csv(folder_path):
    files = glob.glob(os.path.join(folder_path, "*.csv"))
    df_list = []
    
    for f in files:
        try:
            df = pd.read_csv(f, encoding="utf-8")
        except UnicodeDecodeError:
            df = pd.read_csv(f, encoding="latin1")
        df_list.append(df)
        
    merged_df = pd.concat(df_list, ignore_index=True)
    return merged_df


In [5]:
enrolment_df = load_and_merge_csv("data/enrolment")
print("Enrolment shape:", enrolment_df.shape)
enrolment_df.head()


Enrolment shape: (1006029, 7)


,date,state,district,pincode,age_0_5,age_5_17,age_18_greater
0,02-03-2025,Meghalaya,East Khasi Hills,793121,11,61,37
1,09-03-2025,Karnataka,Bengaluru Urban,560043,14,33,39
2,09-03-2025,Uttar Pradesh,Kanpur Nagar,208001,29,82,12
3,09-03-2025,Uttar Pradesh,Aligarh,202133,62,29,15
4,09-03-2025,Karnataka,Bengaluru Urban,560016,14,16,21


In [6]:
demographic_df = load_and_merge_csv("data/demographic")
print("Demographic shape:", demographic_df.shape)
demographic_df.head()


Demographic shape: (2071700, 6)


,date,state,district,pincode,demo_age_5_17,demo_age_17_
0,01-03-2025,Uttar Pradesh,Gorakhpur,273213,49,529
1,01-03-2025,Andhra Pradesh,Chittoor,517132,22,375
2,01-03-2025,Gujarat,Rajkot,360006,65,765
3,01-03-2025,Andhra Pradesh,Srikakulam,532484,24,314
4,01-03-2025,Rajasthan,Udaipur,313801,45,785


In [7]:
biometric_df = load_and_merge_csv("data/biometric")
print("Biometric shape:", biometric_df.shape)
biometric_df.head()


Biometric shape: (1861108, 6)


,date,state,district,pincode,bio_age_5_17,bio_age_17_
0,01-03-2025,Haryana,Mahendragarh,123029,280,577
1,01-03-2025,Bihar,Madhepura,852121,144,369
2,01-03-2025,Jammu and Kashmir,Punch,185101,643,1091
3,01-03-2025,Bihar,Bhojpur,802158,256,980
4,01-03-2025,Tamil Nadu,Madurai,625514,271,815


In [8]:
def clean_base(df):
    df.columns = df.columns.str.lower().str.strip()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    return df

enrolment_df = clean_base(enrolment_df)
demographic_df = clean_base(demographic_df)
biometric_df = clean_base(biometric_df)


In [9]:
key_cols = ["date", "state", "district", "pincode"]

enrolment_df = enrolment_df.drop_duplicates(subset=key_cols)
demographic_df = demographic_df.drop_duplicates(subset=key_cols)
biometric_df = biometric_df.drop_duplicates(subset=key_cols)


In [10]:
enrolment_df["total_enrolments"] = (
    enrolment_df["age_0_5"] +
    enrolment_df["age_5_17"] +
    enrolment_df["age_18_greater"]
)


In [11]:
demo_cols = [c for c in demographic_df.columns if c.startswith("demo_age")]
demographic_df["total_demographic_updates"] = demographic_df[demo_cols].sum(axis=1)


In [12]:
bio_cols = [c for c in biometric_df.columns if c.startswith("bio_age")]
biometric_df["total_biometric_updates"] = biometric_df[bio_cols].sum(axis=1)


In [13]:
print(enrolment_df["total_enrolments"].describe())
print(demographic_df["total_demographic_updates"].describe())
print(biometric_df["total_biometric_updates"].describe())


count    347940.000000
mean          7.783388
std          52.951879
min           1.000000
25%           1.000000
50%           2.000000
75%           5.000000
max        3965.000000
Name: total_enrolments, dtype: float64
count    706906.000000
mean         33.945932
std         192.309526
min           0.000000
25%           3.000000
50%           7.000000
75%          19.000000
max       16942.000000
Name: total_demographic_updates, dtype: float64
count    902201.000000
mean         61.114820
std         229.224196
min           0.000000
25%           3.000000
50%           9.000000
75%          29.000000
max       13381.000000
Name: total_biometric_updates, dtype: float64


In [15]:
os.makedirs("processed", exist_ok=True)

enrolment_df.to_csv("processed/enrolment_clean.csv", index=False)
demographic_df.to_csv("processed/demographic_clean.csv", index=False)
biometric_df.to_csv("processed/biometric_clean.csv", index=False)

print("Clean datasets saved successfully.")


Clean datasets saved successfully.
